In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
font = {'size' : 15}
matplotlib.rc('font', **font)
from skimage import measure
from scipy.spatial import ConvexHull
import os

### LOADING

In [ ]:
field = 0.16
file_name = "data/file_field%1.2f.csv" % field

In [ ]:
data = np.loadtxt(file_name, delimiter=',').astype('int')
nrows, ncols = data.shape

### FUNCTIONS

In [ ]:
def segment_noevents(data, no_event_val=-1):
    no_event_regions = [ r.astype('int') for r in measure.find_contours(data,no_event_val)]
    no_event_regions_sizes = np.array([r.shape[0] for r in no_event_regions])
    sorted_regions_idx = np.argsort(no_event_regions_sizes)
    inner_region = no_event_regions[sorted_regions_idx[-2]]
    outer_region = no_event_regions[sorted_regions_idx[-1]]
    center = inner_region.mean(axis=0)
    return inner_region,  outer_region, center

### EXECUTION

In [ ]:
inner_region,  outer_region, center = segment_noevents(data)

In [ ]:
# mask of the inner (initial) domain, from the convex hull of inner_region
ii, jj = np.meshgrid(np.arange(0,nrows), np.arange(0,ncols), indexing='ij')
pts = np.vstack([ii.flatten(), jj.flatten()])

inner_hull = ConvexHull(inner_region)
inner_mask = np.ones((nrows, ncols), dtype=bool)
for eq in inner_hull.equations:
    eq_mask = np.dot(eq[:-1], pts) + eq[-1] <= 1e-12
    inner_mask[ii.flatten(),jj.flatten()] *= eq_mask

In [ ]:
# interface at each frame = largest contour of (inner domain + pixels switched up to t)
# slow: one find_contours per frame
temp_inner = np.zeros_like(data)
temp_inner[inner_mask] = 1
event_mask = data > - 1
interface_dict = {}
for t in range(0, data.max()+1):
    if(t%5000 == 0):
        print(t)
    temp = np.copy(temp_inner)
    temp[event_mask*(data<=t)] = 1
    regions = measure.find_contours(temp)
    r_sizes = [len(r) for r in regions]
    largest_r = regions[np.argmax(r_sizes)].astype('int')
    interface_dict[t] = largest_r

In [ ]:
# polar interpolation on a regular theta grid
n_th = 2**11
theta_new = np.linspace(-np.pi, np.pi, n_th)
interface_data_interp = []
for t, h in interface_dict.items():
    theta = np.arctan2(h[:, 0] - center[0], h[:, 1]-center[1])
    rho = np.sqrt((h[:, 0] - center[0])**2+(h[:, 1]-center[1])**2)
    rho_new =  np.interp(theta_new, theta, rho, period=np.pi*2)
    interface_data_interp.append(rho_new)
interface_data_interp = np.array(interface_data_interp)

In [ ]:
# the interface cannot recede
interface = [np.copy(interface_data_interp[0])]
for t in range(1, interface_data_interp.shape[0]):
    new_interface = np.copy(interface_data_interp[t])
    mask_receeded = new_interface-interface[-1] < 0
    new_interface[mask_receeded] = interface[-1][mask_receeded]
    interface.append(new_interface)
interface = np.array(interface)

In [ ]:
#dest_folder = "/data/biophys/schimmenti/earthquakes/MOKE_interfaces/"
dest_folder = "interface_data/"
os.makedirs(dest_folder, exist_ok=True)
np.savez("%sinterface_field=%1.2f.npz" % (dest_folder,field), interface=interface, theta_new=theta_new, center=center)